<a href="https://colab.research.google.com/github/PujaSridhar/finsight/blob/main/notebooks/03_compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight · 3. Compare against the alternatives

The same 100 held-out releases (companies never seen in training) go to four systems with the same instruction:
base Mistral-7B, base Mistral-7B with two retrieved worked examples (RAG), FinSight, and a large hosted model.
Each local system takes roughly an hour on a T4, one request at a time so latency is real. Predictions are saved
as they go; re-running resumes.

In [ ]:
# Clone the repo and install dependencies (~2 min)
REPO_BRANCH = "main"   # to try a branch before merging it, put its name here
!git clone -q -b $REPO_BRANCH https://github.com/PujaSridhar/finsight /content/finsight 2>/dev/null || (cd /content/finsight && git pull -q)
%cd /content/finsight
!pip install -q -r requirements.txt

In [ ]:
# Keep data, checkpoints and results on Google Drive so a disconnect doesn't lose them
# and every notebook sees the same dataset.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/finsight/outputs /content/drive/MyDrive/finsight/results
!mkdir -p /content/drive/MyDrive/finsight/data
!rm -rf outputs results data && ln -s /content/drive/MyDrive/finsight/outputs outputs && ln -s /content/drive/MyDrive/finsight/results results && ln -s /content/drive/MyDrive/finsight/data data

In [ ]:
# Mistral-7B-Instruct-v0.3 is gated: accept its terms at
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3, then paste a read token here
# (or store it as the Colab secret HF_TOKEN).
from huggingface_hub import login
login()

In [ ]:
!python -m finsight.compare --system base

In [ ]:
!python -m finsight.compare --system rag

In [ ]:
!python -m finsight.compare --system finetuned --adapter outputs/finsight-earnings/final

## Optional: the large hosted model
Needs an Anthropic API key; 100 releases cost roughly a dollar at the prices in `configs/pricing.yaml`.

In [ ]:
# import os; os.environ["ANTHROPIC_API_KEY"] = "..."   # or use a Colab secret
# !pip install -q anthropic
# !python -m finsight.compare --system hosted

## Comparison table (copy into the README)
Put what the GPU would cost you per hour, so local cost per release is a real number.

In [ ]:
GPU_HOURLY_USD = None   # e.g. your cloud provider's on-demand price for the GPU you ran on
!python -m finsight.compare --report {"--gpu-hourly-usd " + str(GPU_HOURLY_USD) if GPU_HOURLY_USD else ""}
from IPython.display import Markdown, display
display(Markdown(open("results/compare/COMPARISON.md").read()))

## Error analysis: what does each system get wrong?

In [ ]:
import json
for system in ["base", "finetuned"]:
    preds = [json.loads(l) for l in open(f"results/compare/predictions/{system}_earnings.jsonl")]
    wrong = [p for p in preds if not p["correct"]]
    print(f"== {system}: {len(wrong)} of {len(preds)} releases have a wrong field")
    for p in wrong[:4]:
        print(p["ticker"], p["filed"], "\nGOLD :", json.dumps(p["label"]), "\nMODEL:", p["response"][:300], "\n")